# LC 102 — Binary Tree Level Order Traversal
**Difficulty:** Medium | **Category:** BFS on Trees
**Pattern:** Level-by-Level BFS with deque

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Use a deque. At each BFS step,
snapshot <code>len(queue)</code> — that count tells you exactly
how many nodes belong to the current level. Drain exactly that
many nodes, collect their values, then move on to the next level.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return *the level order traversal
of its nodes' values* (i.e., from left to right, level by level).

**Constraints:**
- The number of nodes in the tree is in the range `[0, 2000]`.
- `-1000 <= Node.val <= 1000`

## What This Is Actually Asking

Imagine reading a family tree one generation at a time.
You want all grandparents first, then all parents, then all children.
Each generation goes left-to-right.
The result is a list of lists — one inner list per level.
An empty tree returns an empty list.

## Walk Through an Example by Hand

Tree: `[3, 9, 20, None, None, 15, 7]`

```
        3
       / \
      9   20
         /  \
        15    7
```

**Step 1:** queue = [3], level_size = 1
- Pop 3 → level = [3]
- Add children: 9, 20
- result = [[3]]

**Step 2:** queue = [9, 20], level_size = 2
- Pop 9 → level = [9] (no children)
- Pop 20 → level = [9, 20] (add 15, 7)
- result = [[3], [9, 20]]

**Step 3:** queue = [15, 7], level_size = 2
- Pop 15 → level = [15]
- Pop 7  → level = [15, 7]
- result = [[3], [9, 20], [15, 7]]

## The Picture

```
         3          <- Level 0
        / \
       9   20       <- Level 1
          /  \
         15    7    <- Level 2

BFS Queue States:
┌─────────────────────────────────────────────────┐
│ Start      │ [3]                                │
│ After L0   │ [9, 20]   level_size=1 → pop 1    │
│ After L1   │ [15, 7]   level_size=2 → pop 2    │
│ After L2   │ []        level_size=2 → pop 2    │
└─────────────────────────────────────────────────┘

Key: level_size = len(queue) BEFORE draining the level
     This freezes the boundary between levels.
```

## When To Use This Pattern

- When you see "level by level" or "layer by layer" → BFS deque.
- When you need to process nodes at the same depth together
  → snapshot `len(queue)` before each level.
- When output must be grouped per depth → collect into sublists.
- When shortest path in an unweighted graph is needed → same BFS.
- When you need to distinguish which generation a node belongs to
  → level-size trick.

## The Approach

Push the root into a deque (if it exists).
While the deque is not empty, record how many nodes are currently
in it — that is the level size.
Pop exactly that many nodes left-to-right, collect their values
into a level list, and push each node's children into the deque.
Append the level list to the result and repeat.

In [ ]:
from collections import deque        # BFS queue
from typing import Optional, List    # type hints


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    """Build a binary tree from a level-order list.
    None in the list means no node at that position.
    """
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    """Run test cases against func(root) -> List[List[int]]."""
    cases = [
        # (vals_list, expected)
        ([3, 9, 20, None, None, 15, 7],
         [[3], [9, 20], [15, 7]]),
        ([1],
         [[1]]),                        # single node
        ([],
         []),                           # empty tree
        ([1, 2, 3, 4, 5, 6, 7],
         [[1], [2, 3], [4, 5, 6, 7]]), # full tree
        ([1, None, 2, None, 3],
         [[1], [2], [3]]),              # right-skewed
    ]
    passed = 0
    for vals, expected in cases:
        root = make_tree(vals)
        result = func(root)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"{status} | input={vals}")
            print(f"        expected={expected}")
            print(f"        got     ={result}")
    print(f"\n{passed}/{len(cases)} tests passed.")

In [ ]:
def level_order(root: Optional[TreeNode]) -> List[List[int]]:
    """
    Return level-order traversal of binary tree.

    Args:
        root: Root of the binary tree (may be None).

    Returns:
        List of levels; each level is a list of node values
        from left to right.

    Approach:
        BFS with deque. Snapshot level_size = len(queue)
        before draining each level.
    """
    pass


# --- Debug prints (remove pass above before running) ---

# Basic example
t1 = make_tree([3, 9, 20, None, None, 15, 7])
print(level_order(t1))   # expected [[3], [9, 20], [15, 7]]

# Single node
t2 = make_tree([1])
print(level_order(t2))   # expected [[1]]

# Empty tree
print(level_order(None)) # expected []

# Full tree
t4 = make_tree([1, 2, 3, 4, 5, 6, 7])
print(level_order(t4))   # expected [[1],[2,3],[4,5,6,7]]

# Right-skewed
t5 = make_tree([1, None, 2, None, 3])
print(level_order(t5))   # expected [[1],[2],[3]]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(level_order)

## Complexity

| Approach     | Time   | Space  |
|--------------|--------|--------|
| Brute force (recursion w/ depth param) | O(n) | O(n) |
| Optimal BFS (deque + level_size)       | O(n) | O(n) |

- **Time O(n):** every node is enqueued and dequeued once.
- **Space O(n):** the deque holds at most one full level at a time;
  the widest level of a complete tree has n/2 nodes → O(n).

## Real World Connection

At **Citi**, org-chart tooling enumerates employees generation by
generation — every direct report at the same management level is
processed together before moving deeper, which is exactly level
order traversal.
On **AWS**, Config's resource dependency graph is explored layer
by layer to find all resources affected at each blast radius.
In **data engineering**, DAG schedulers like Airflow emit tasks in
topological layers; within each layer tasks share the same minimum
distance from the source, matching the level-order property.
Any breadth-first sweep of a hierarchy — file system, DNS zone
delegation, or Kubernetes namespace tree — uses this same pattern.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra